In [48]:
import numpy as np
import pandas as pd

float_list = ["Durée moyenne du trajet", "Retard moyen des trains en retard au départ", "Retard moyen des trains en retard à l'arrivée","Retard moyen trains en retard > 15 (si liaison concurrencée par vol)","Retard moyen de tous les trains à l'arrivée",
"Retard moyen de tous les trains au départ","Prct retard pour causes externes","Prct retard pour cause infrastructure","Prct retard pour cause gestion trafic","Prct retard pour cause matériel roulant",
"Prct retard pour cause gestion en gare et réutilisation de matériel","Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"]

int_list = ["Nombre de circulations prévues","Nombre de trains annulés","Nombre de trains en retard à l'arrivée","Nombre de trains en retard au départ","Nombre trains en retard > 30min","Nombre trains en retard > 60min","Nombre trains en retard > 15min"]

check_if_zero = ["Durée moyenne du trajet","Nombre de circulations prévues"]

# create dataframe and clear row and duplicate row
data = pd.read_csv('project_dataset.csv', on_bad_lines='warn', sep=';')
data.drop(['Commentaire annulations','Commentaire retards au départ',"Commentaire retards à l'arrivée","Service"],axis=1,inplace=True)

# Strips the strings, replaces spaces and / to -, and formats to year-month-days dates.
data['Date'] = data['Date'].str.strip().str.replace(' ',"-").str.replace('/','-')
data['Date'] = pd.to_datetime(data['Date'], format='mixed')

# Creates year and month dataframes.
YearDf = data['Date'].dt.year
monthDf = data['Date'].dt.month

## change value if case is empty
for element in data:
    if element in float_list or element in int_list:
        data[element] = data[element].astype(str).replace(['', 'NC', 'Non communiqué', 'N/A','-',' '], '0')
    else:
        data[element] = data[element].replace('', np.nan)
        data[element] = data[element].replace('NC', np.nan)
        data[element] = data[element].replace('Non communiqué', np.nan)
        data[element] = data[element].replace('N/A', np.nan)
        data[element] = data[element].replace('-', np.nan)

# format line to dataframe
for element in data:
    if element in float_list:
        data[element] = data[element].astype(str).str.replace('min', '').str.replace(',', '.').str.replace('"', '').str.replace('%', '').str.replace('-', '').str.strip().astype(float).round(2)
    elif element in int_list:
        data[element] = data[element].astype(str).str.replace('min', '').str.replace(',', '.').str.replace('"', '').str.replace('%', '').str.replace('-', '').str.strip()
        data[element] = pd.to_numeric(data[element], errors='coerce').fillna(0).astype(int)
    elif element == "Gare de départ" or element == "Gare d'arrivée":
        data[element] = data[element].str.upper().str.replace("SAINT","ST").str.replace("TGV","").str.replace('0', '').str.replace('1', '').str.replace('2', '').str.replace('3', '').str.replace('4', '').str.replace('5', '').str.replace('6', '').str.replace('7', '').str.replace('8', '').str.replace('9', '').str.strip()

# check if the value is 0.0 and drop the row if it is          
for index in check_if_zero:
    data = data[data[index] != 0]

# check all line if it's ok and drop the row if not
data = data[data["Nombre de circulations prévues"] >= data["Nombre de trains annulés"]]
data = data[data["Nombre de circulations prévues"] >= data["Nombre de trains en retard au départ"]]
data = data[data["Nombre de circulations prévues"] >= data["Nombre de trains en retard à l'arrivée"]]
data = data[data["Nombre de circulations prévues"] >= data["Nombre trains en retard > 30min"]]
data = data[data["Nombre de circulations prévues"] >= data["Nombre trains en retard > 60min"]]

data = data[data["Gare de départ"].notna()]
data = data[data["Gare d'arrivée"].notna()]



# Create new csv file with cleaned data
data.to_csv('cleaned_dataset.csv', index=False)


In [49]:
#  add cleaned data to a new dataframe and display info
cleaned_data = pd.read_csv('cleaned_dataset.csv')
cleaned_data['Date'] = pd.to_datetime(cleaned_data['Date'], format='ISO8601')
cleaned_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 11627 entries, 0 to 11626
Data columns (total 22 columns):
 #   Column                                                                                       Non-Null Count  Dtype         
---  ------                                                                                       --------------  -----         
 0   Date                                                                                         11569 non-null  datetime64[us]
 1   Gare de départ                                                                               11623 non-null  str           
 2   Gare d'arrivée                                                                               11615 non-null  str           
 3   Durée moyenne du trajet                                                                      11397 non-null  float64       
 4   Nombre de circulations prévues                                                               11627 non-null  int64         

In [50]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(12,6))
# graph_data = cleaned_data.sort_values(["Gare d'arrivée"])

# sns.countplot(data=cleaned_data, x='Durée moyenne du trajet', hue="Gare d'arrivée")
# plt.xticks(rotation=90)
# plt.show()